In [0]:
# MAGIC %md
# ============================================================
# NOTEBOOK   : gold_sales_summary
# PURPOSE    : Monthly revenue KPIs for Power BI dashboard
# SOURCE     : silver.orders, silver.order_payments
# DESTINATION: fincompliance_catalog.gold.sales_summary
# METRICS:
#   - Total orders per month
#   - Total revenue per month
#   - Average order value
#   - Unique customers per month
#   - Month over month growth
# ============================================================

In [0]:
%run ../configs/config

In [0]:
authenticate_spark(spark)
set_catalog(spark)

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    sum as spark_sum,
    avg,
    round as spark_round,
    concat,
    lit,
    when,
    lag,
    current_timestamp
)
from pyspark.sql.window import Window

In [0]:
# ============================================================
# CELL 5 - READ FROM SILVER
# Gold layer reads from SILVER not bronze
# This is the key difference between layers
# ============================================================

df_orders = spark.table(f"{SILVER_DB}.orders")
df_payments = spark.table(f"{SILVER_DB}.order_payments")

print("=" * 60)
print("SILVER TABLES LOADED")
print("=" * 60)
print(f"orders          : {df_orders.count():,} rows")
print(f"order_payments  : {df_payments.count():,} rows")

print("\norders columns:")
for c in df_orders.columns:
    print(f"  {c}")

print("\norder_payments columns:")
for c in df_payments.columns:
    print(f"  {c}")

print("\nOrder status breakdown:")
df_orders.groupBy("order_status").count() \
    .orderBy("count", ascending=False).show()

In [0]:
# ============================================================
# CALCULATE MONTHLY SALES SUMMARY
# Join orders with payments
# Filter only DELIVERED orders for revenue accuracy
# Group by year and month
# ============================================================

df_sales_summary = (
    df_orders
    .filter(col("order_status") == "delivered")
    .join(
        df_payments.select("order_id", "payment_value"),
        on="order_id",
        how="left"
    )
    .groupBy("order_year", "order_month")
    .agg(
        count("order_id").alias("total_orders"),
        spark_round(spark_sum("payment_value"), 2)
            .alias("total_revenue"),
        spark_round(avg("payment_value"), 2)
            .alias("avg_order_value"),
        countDistinct("customer_id")
            .alias("unique_customers")
    )
    .orderBy("order_year", "order_month")
)

print("Monthly sales summary:")
df_sales_summary.show(30, truncate=False)

print(f"\nTotal months : {df_sales_summary.count()}")

In [0]:
# ============================================================
# HANDLE NULL REVENUE AND ADD GROWTH METRICS
# Some orders have no payment record - replace null with 0
# Add month over month revenue growth percentage
# ============================================================

# Handle null revenue
df_sales_summary = df_sales_summary \
    .withColumn(
        "total_revenue",
        when(col("total_revenue").isNull(), 0)
        .otherwise(col("total_revenue"))
    ) \
    .withColumn(
        "avg_order_value",
        when(col("avg_order_value").isNull(), 0)
        .otherwise(col("avg_order_value"))
    )

# Add year_month column for easier Power BI sorting
df_sales_summary = df_sales_summary \
    .withColumn(
        "year_month",
        concat(
            col("order_year"),
            lit("-"),
            when(col("order_month") < 10,
                 concat(lit("0"), col("order_month")))
            .otherwise(col("order_month").cast("string"))
        )
    )

# Add month over month growth using window function
window_spec = Window.orderBy("order_year", "order_month")

df_sales_summary = df_sales_summary \
    .withColumn(
        "previous_month_revenue",
        lag("total_revenue", 1).over(window_spec)
    ) \
    .withColumn(
        "revenue_growth_pct",
        when(
            col("previous_month_revenue").isNull() |
            (col("previous_month_revenue") == 0),
            None
        )
        .otherwise(
            spark_round(
                ((col("total_revenue") -
                  col("previous_month_revenue")) /
                 col("previous_month_revenue")) * 100, 2
            )
        )
    )

print("Sales summary with growth metrics:")
df_sales_summary \
    .select(
        "year_month",
        "total_orders",
        "total_revenue",
        "previous_month_revenue",
        "revenue_growth_pct"
    ) \
    .show(30, truncate=False)

print("Done - Growth metrics added")

In [0]:
# ============================================================
# FILTER OUT LOW VOLUME MONTHS FOR ACCURATE GROWTH
# Months with less than 50 orders create misleading
# growth percentages due to tiny denominators
# Add a flag instead of removing data
# ============================================================

df_sales_summary = df_sales_summary \
    .withColumn(
        "is_reliable_month",
        when(col("total_orders") < 50, 0)
        .otherwise(1)
    )

print("Reliable vs unreliable months:")
df_sales_summary \
    .groupBy("is_reliable_month") \
    .count() \
    .show()

print("\nGrowth metrics for RELIABLE months only:")
df_sales_summary \
    .filter(col("is_reliable_month") == 1) \
    .select(
        "year_month",
        "total_orders",
        "total_revenue",
        "revenue_growth_pct"
    ) \
    .show(30, truncate=False)

print("Done - Reliability flag added")

In [0]:
# ============================================================
# ADD GOLD AUDIT COLUMN
# ============================================================

df_sales_summary = df_sales_summary \
    .withColumn("gold_updated_at", current_timestamp())

print("Final columns:")
for col_name in df_sales_summary.columns:
    print(f"  {col_name}")
print(f"Total rows : {df_sales_summary.count()}")

In [0]:
# ============================================================
# WRITE TO GOLD
# ============================================================

(
    df_sales_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_DB}.sales_summary")
)

print(f"Written to {GOLD_DB}.sales_summary")
print(f"Rows : {df_sales_summary.count()}")

In [0]:
# ============================================================
# VERIFICATION
# ============================================================
print("=" * 60)
print("GOLD.SALES_SUMMARY VERIFICATION")
print("=" * 60)

df_gold = spark.table(f"{GOLD_DB}.sales_summary")
print(f"Total months : {df_gold.count()}")
print(f"Total columns: {len(df_gold.columns)}")

print("\nColumns:")
for c in df_gold.columns:
    print(f"  {c}")

print("\nTotal revenue across all months:")
total_rev = df_gold.agg(
    spark_round(spark_sum("total_revenue"), 2)
).collect()[0][0]
print(f"  R$ {total_rev:,.2f}")

print("\nTotal orders across all months:")
total_orders = df_gold.agg(
    spark_sum("total_orders")
).collect()[0][0]
print(f"  {total_orders:,}")

print("\nBest month by revenue:")
df_gold.orderBy(col("total_revenue").desc()).show(1, truncate=False)

print("=" * 60)
print("gold.sales_summary verification complete!")
print("=" * 60)